In [ ]:
import torch
import torch.nn as nn

from torch.optim.swa_utils import AveragedModel, update_bn
from torch.utils.tensorboard import SummaryWriter

from detection import *

In [ ]:
device = get_device()
device

In [ ]:
data_path = './data/'
image_dir = os.path.join(data_path, 'images')
gt_dir = os.path.join(data_path, 'gt.csv')

In [ ]:
train_loader, val_loader = prepare_dataloaders(
    image_dir,
    gt_dir,
    batch_size=32,
    split=(0.9, 0.1),
    sigma=3,
)

In [ ]:
model = UNet()
optimizer = torch.optim.AdamW(model.parameters())

checkpoint = torch.load('./checkpoints/pretrained_model_checkpoint_sigma3.pt', weights_only=True)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

swa_model = AveragedModel(model)

new_lr = 1e-6
for param_group in optimizer.param_groups:
    param_group['lr'] = new_lr
for state in optimizer.state.values():
    for k, v in state.items():
        if isinstance(v, torch.Tensor):
            state[k] = v.to(device)

epochs = checkpoint['epoch']
loss = checkpoint['loss']
print(f'Resuming training of model pretrained on {epochs=} with final {loss=}')

n_epochs = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=n_epochs * len(train_loader), eta_min=1e-8,
)
logger = SummaryWriter(log_dir='./logs/pretrained_model1')
trainer = Trainer(
    model=model.to(device),
    swa_model=swa_model,
    criterion=weighted_mse_loss,
    optimizer=optimizer,
    scheduler=scheduler,
    logger=logger,
    device=device,
)

In [ ]:
val_losses = trainer.train(
    train_loader, val_loader, n_epochs=n_epochs,
)

In [ ]:
swa_model = trainer.swa_model.cpu()
update_bn(train_loader, swa_model)

In [ ]:
swa_model.eval()
loss_total = 0
for inputs, targets in val_loader:
    inputs = inputs#.to(device)
    targets = targets#.to(device)
    output = swa_model(inputs)
    target_coo = heatmaps_to_coords(targets.cpu())
    output_coo = heatmaps_to_coords(output.cpu())
    loss_total += nn.functional.mse_loss(target_coo, output_coo)
loss_total /= len(val_loader)
loss_total = loss_total.cpu().item()
loss_total

In [ ]:
torch.save(swa_model.state_dict(), "./models/model_swa_4.3.pt")